# Question 1

In [64]:
import numpy as np
import matplotlib.pyplot as plt
import galsim
import galsim.hsm
from pathlib import Path
 
 
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
 
STAMPS_FILE = DATA_DIR / "galaxy_stamps.fits"

## Question 1a

In [65]:
def _pixel_grids(image: galsim.Image):
    """Return (x, y) arrays of pixel coordinates relative to image centre."""
    arr = image.array                          # shape (N, N)
    ny, nx = arr.shape
    cx, cy = (nx - 1) / 2.0, (ny - 1) / 2.0  # centre in pixel coords
    x_idx = np.arange(nx) - cx
    y_idx = np.arange(ny) - cy
    x, y = np.meshgrid(x_idx, y_idx)          # both shape (ny, nx)
    return x, y

In [66]:
def unweighted_ellipticity(image: galsim.Image):
    """
    Estimate (e1, e2) from unweighted second-order moments.
 
    Q_ij = sum_pq [ I(p,q) * delta_i(p,q) * delta_j(p,q) ]
    e1 = (Q11 - Q22) / (Q11 + Q22)
    e2 =  2 * Q12    / (Q11 + Q22)
 
    where delta_x = x - <x>, delta_y = y - <y> (centroid-subtracted).
 
    Returns
    -------
    (e1, e2) : floats, or (nan, nan) if the denominator is non-positive.
    """
    arr = image.array.astype(float)
    x, y = _pixel_grids(image)
 
    I_sum = arr.sum()
    if I_sum <= 0:
        return np.nan, np.nan
 
    # Centroid (first moments)
    xbar = (arr * x).sum() / I_sum
    ybar = (arr * y).sum() / I_sum
 
    dx = x - xbar
    dy = y - ybar
 
    # Second moments
    Q11 = (arr * dx * dx).sum()
    Q22 = (arr * dy * dy).sum()
    Q12 = (arr * dx * dy).sum()
 
    denom = Q11 + Q22
    if denom <= 0:
        return np.nan, np.nan
 
    e1 = (Q11 - Q22) / denom
    e2 = 2.0 * Q12 / denom
    return e1, e2

In [67]:
def _gaussian_weight(x, y, sigma):
    """Circular Gaussian weight function W(x, y) = exp(-(x^2+y^2)/(2*sigma^2))."""
    return np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
 
 
def weighted_ellipticity(image: galsim.Image, fwhm_px: float = 4.0):
    """
    Estimate (e1, e2) from weighted second-order moments.
 
    The weight function is a circular Gaussian with the given FWHM.
    sigma = FWHM / (2 * sqrt(2 * ln2))
 
    Weighted moments:
        Q_ij^W = sum_pq [ W(p,q) * I(p,q) * delta_i * delta_j ]
                 / sum_pq [ W(p,q) * I(p,q) ]
 
    Ellipticity definition is identical to the unweighted case.
 
    The centroid is first estimated from the unweighted moments and the weight
    function is centred there.  This is a single-pass approximation; for
    production code one would iterate.
 
    Returns
    -------
    (e1, e2) : floats, or (nan, nan) on failure.
    """
    arr = image.array.astype(float)
    x, y = _pixel_grids(image)
 
    sigma = fwhm_px / (2.0 * np.sqrt(2.0 * np.log(2.0)))
 
    # --- Step 1: unweighted centroid ---
    I_sum = arr.sum()
    if I_sum <= 0:
        return np.nan, np.nan
    xbar = (arr * x).sum() / I_sum
    ybar = (arr * y).sum() / I_sum
 
    dx = x - xbar
    dy = y - ybar
 
    # --- Step 2: Gaussian weight centred on unweighted centroid ---
    W = _gaussian_weight(dx, dy, sigma)
 
    WI = W * arr
    WI_sum = WI.sum()
    if WI_sum <= 0:
        return np.nan, np.nan
 
    # --- Step 3: weighted second moments ---
    Q11 = (WI * dx * dx).sum()
    Q22 = (WI * dy * dy).sum()
    Q12 = (WI * dx * dy).sum()
 
    denom = Q11 + Q22
    if denom <= 0:
        return np.nan, np.nan
 
    e1 = (Q11 - Q22) / denom
    e2 = 2.0 * Q12 / denom
    return e1, e2


In [68]:
def hsm_ellipticity(image: galsim.Image):
    """
    Estimate (e1, e2) using GalSim's HSM adaptive-moments algorithm.
 
    Returns
    -------
    (e1, e2) : floats, or (nan, nan) if HSM fails.
    """
    try:
        result = galsim.hsm.FindAdaptiveMom(image)
        return result.observed_shape.g1, result.observed_shape.g2
    except galsim.hsm.HSMError:
        return np.nan, np.nan

In [69]:
def measure_all_ellipticities(stamps_path: Path):
    """
    Load all galaxy stamps and compute (e1, e2) for each estimator.
 
    Returns
    -------
    dict with keys 'unweighted', 'weighted', 'hsm', each mapping to
    an ndarray of shape (N, 2) containing (e1, e2) per galaxy.
    """
    galaxy_ims = galsim.fits.readMulti(str(stamps_path))
    n_gals = len(galaxy_ims)
    print(f"Loaded {n_gals} galaxy stamps from {stamps_path}")
 
    e_unw = np.full((n_gals, 2), np.nan)
    e_wgt = np.full((n_gals, 2), np.nan)
    e_hsm = np.full((n_gals, 2), np.nan)
 
    for i, im in enumerate(galaxy_ims):
        e_unw[i] = unweighted_ellipticity(im)
        e_wgt[i] = weighted_ellipticity(im, fwhm_px=4.0)
        e_hsm[i] = hsm_ellipticity(im)
 
    results = {
        "unweighted": e_unw,
        "weighted":   e_wgt,
        "hsm":        e_hsm,
    }
 
    # Summary statistics
    for name, arr in results.items():
        valid = np.isfinite(arr[:, 0])
        n_valid = valid.sum()
        print(
            f"  {name:12s}: {n_valid}/{n_gals} valid | "
            f"<e1> = {arr[valid, 0].mean():.4f} ± {arr[valid, 0].std():.4f} | "
            f"<e2> = {arr[valid, 1].mean():.4f} ± {arr[valid, 1].std():.4f}"
        )
 
    return results

In [70]:
def plot_ellipticity_comparison(results: dict, output_dir: Path):
    """
    Produce scatter plots and histograms comparing the three estimators.
    Saves figures to output_dir.
    """
    methods  = ["unweighted", "weighted", "hsm"]
    colours  = {"unweighted": "steelblue", "weighted": "tomato", "hsm": "seagreen"}
    labels   = {"unweighted": "Unweighted moments",
                 "weighted":   "Weighted moments (Gaussian, FWHM=4px)",
                 "hsm":        "HSM (Hirata-Seljak-Mandelbaum)"}
 
    # ---- 1. Scatter: e1 vs e2 per method ------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, method in zip(axes, methods):
        arr = results[method]
        valid = np.isfinite(arr[:, 0])
        ax.scatter(arr[valid, 0], arr[valid, 1],
                   s=4, alpha=0.5, color=colours[method])
        ax.axhline(0, lw=0.7, color="k", ls="--")
        ax.axvline(0, lw=0.7, color="k", ls="--")
        ax.set_xlabel(r"$e_1$", fontsize=13)
        ax.set_title(labels[method], fontsize=10)
        n_valid = valid.sum()
        n_total = len(valid)
        ax.text(0.97, 0.97, f"N={n_valid}/{n_total}",
                transform=ax.transAxes, ha="right", va="top", fontsize=9)
    axes[0].set_ylabel(r"$e_2$", fontsize=13)
    fig.suptitle("Ellipticity estimates per method: $e_1$ vs $e_2$", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_scatter_e1_e2.png"
    fig.savefig(out, dpi=400, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")
 
    # ---- 2. Histograms of e1 and e2 -----------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
    bins = np.linspace(-1, 1, 50)
    for col, method in enumerate(methods):
        arr = results[method]
        valid = np.isfinite(arr[:, 0])
        for row, comp in enumerate(["e1", "e2"]):
            ax = axes[row, col]
            data = arr[valid, row]
            ax.hist(data, bins=bins, color=colours[method], alpha=0.7, density=True)
            ax.axvline(data.mean(), color="k", lw=1.5, ls="--",
                       label=f"mean={data.mean():.3f}")
            ax.set_title(f"{labels[method]}\n${comp}$  (σ={data.std():.3f})", fontsize=9)
            ax.legend(fontsize=8)
            if row == 1:
                ax.set_xlabel(f"${comp}$", fontsize=12)
            if col == 0:
                ax.set_ylabel("Density", fontsize=11)
    fig.suptitle("Ellipticity distributions per method", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_histograms.png"
    fig.savefig(out, dpi=400, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")
 
    # ---- 3. Direct comparison: each method vs HSM ---------------------------
    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    e_hsm = results["hsm"]
    hsm_valid = np.isfinite(e_hsm[:, 0])
 
    compare_pairs = [
        ("unweighted", "e1"),
        ("unweighted", "e2"),
        ("weighted",   "e1"),
        ("weighted",   "e2"),
    ]
    for ax, (method, comp) in zip(axes.flat, compare_pairs):
        arr = results[method]
        idx = 0 if comp == "e1" else 1
        both_valid = hsm_valid & np.isfinite(arr[:, 0])
        x_vals = e_hsm[both_valid, idx]
        y_vals = arr[both_valid, idx]
 
        ax.scatter(x_vals, y_vals, s=4, alpha=0.45, color=colours[method])
        lim = 0.9
        ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, label="1:1")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel(f"HSM  ${comp}$", fontsize=12)
        ax.set_ylabel(f"{labels[method].split('(')[0].strip()}  ${comp}$", fontsize=11)
        ax.legend(fontsize=8)
 
        # Pearson r
        r = np.corrcoef(x_vals, y_vals)[0, 1]
        ax.text(0.04, 0.95, f"r = {r:.3f}", transform=ax.transAxes,
                fontsize=9, va="top")
 
    fig.suptitle("Method comparison vs HSM", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_comparison_vs_hsm.png"
    fig.savefig(out, dpi=400, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")

In [71]:
def main():
    results = measure_all_ellipticities(STAMPS_FILE)
    plot_ellipticity_comparison(results, OUTPUT_DIR)
 
    # Persist results for use in Q2
    np.save(OUTPUT_DIR / "ellipticities_unweighted.npy", results["unweighted"])
    np.save(OUTPUT_DIR / "ellipticities_weighted.npy",   results["weighted"])
    np.save(OUTPUT_DIR / "ellipticities_hsm.npy",        results["hsm"])
    print("Ellipticity arrays saved to outputs/.")
    return results
 
 
if __name__ == "__main__":
    main()

Loaded 100000 galaxy stamps from data/galaxy_stamps.fits
  unweighted  : 97364/100000 valid | <e1> = -0.0077 ± 2.8059 | <e2> = -0.0897 ± 26.4164
  weighted    : 99998/100000 valid | <e1> = 0.0000 ± 0.0263 | <e2> = -0.0000 ± 0.0254
  hsm         : 100000/100000 valid | <e1> = 0.0001 ± 0.0572 | <e2> = 0.0001 ± 0.0573
  Saved outputs/q1_scatter_e1_e2.png
  Saved outputs/q1_histograms.png
  Saved outputs/q1_comparison_vs_hsm.png
Ellipticity arrays saved to outputs/.
